In [ ]:
import sys
import os
from pathlib import Path  # noqa: F401

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

import zipfile  # noqa: E402
import numpy as np  # noqa: E402
import numpy.lib.format as npformat  # noqa: E402
import pandas as pd  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import seaborn as sns  # noqa: E402
import mne  # noqa: E402
from scipy.stats import zscore  # noqa: E402
from sklearn.decomposition import PCA  # noqa: E402

from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    MusicTypeVariants,
    ExperimentNames,
    PreprocessedDataVariants,
)
from src.definitions.constants import ProjectPaths  # noqa: E402

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
mne.set_log_level("ERROR")
print("Setup complete.")

# ASSR Wavelet Power — Channel-PCA Time-Frequency Map

Reduce the stimulus-locked **wavelet power** to a single **time-frequency (TF) map
per participant** by collapsing the channel dimension with **PCA over channels**,
following the same data-access workflow as
`assr_stimulus_aligned_inspection.ipynb`.

Pipeline (per subject, per project convention on the **Placebo** condition):

1. **Trial average.** Epoch the wavelet power around every `fam+` onset and
   average over stimuli. The epoch keeps an **initial pad before onset** (negative
   time) and a post-onset length equal to the **shortest inter-onset interval**, so
   no epoch ever overlaps the next stimulus (longer intervals are trimmed to the
   shortest gap; edge windows that don't fit the recording are skipped).
2. **Channel PCA.** Treat each `(freq, time)` bin as an observation and each
   channel as a variable. Fit PCA over the `(n_freqs*n_times, n_channels)` matrix
   and keep the **first component's score map**, reshaped to `(n_freqs, n_times)`.
   This reduces the channel dimension to the single dominant spatial mode.
3. **Plot** the first-component TF map, per participant and averaged over
   participants.

The stimulus alignment (see `00-preprocessing/stimulus_alignment.ipynb`) splices
every recording so each onset lands at the **same sample index** in all
participants — one shared onsets array serves every subject.

> **Small-subset notebook by design.** The wavelet file is ~49 GB, read **lazily**
> — only the selected subjects are decompressed, and each channel block is reduced
> to its small `(n_freqs, win)` trial average on the fly (all channels are kept
> because PCA needs them). Keep `SUBJECT_INDICES` to the *lowest* indices: the
> streaming reader scans from the start, so `[0, 1, 2]` is far cheaper than
> `[0, 7, 14]`.

## Configuration

In [ ]:
EXPERIMENT = ExperimentNames.ASSR
CONDITION = ConditionVariants.PLACEBO   # default per project convention
MUSIC_TYPE = MusicTypeVariants.ASSR

# ── Subjects to reduce (all channels are used by the PCA) ─────────────────────
# Subject indices into the concatenated array (CONCATENATED_PERSON_INDEX). Keep
# these LOW — the lazy reader scans from the start, so [0, 1, 2] only decompresses
# the first 3 of 15 subject blocks.
SUBJECT_INDICES = [3, 4, 5,]

# ── Stimulus-locked epoch window ─────────────────────────────────────
# Initial pad kept BEFORE each onset (negative time), in seconds. The post-onset
# length is derived from the data (shortest inter-onset interval) in the subset
# cell below, so longer intervals are trimmed and epochs never overlap.
PRE_PAD_S = 0.1

ASSR_FREQ = 40.0           # expected steady-state frequency (Hz)
SFREQ = 250.0              # sampling rate of the concatenated / wavelet data

# Z-score each channel's per-frequency power against the WHOLE recording before
# epoching (matches the inspection notebook): removes the 1/f tilt so the PCA is
# not dominated by absolute low-frequency power. Set False to run PCA on raw power.
ZSCORE_VS_RECORDING = True

# ── Wavelet cache descriptor (matches the stored filename) ─────────────────
WAVELET_FREQ_SIG = "1.000_50.000_50"   # freqs[0]_freqs[-1]_n_freqs
N_WAVELET_FREQS = 50

# ── Plot saving ────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "00-preprocessing"
    / "plots"
    / "assr_wavelet_pca"
    / f"{CONDITION.value}_{MUSIC_TYPE.value}"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Group          : {CONDITION.value} / {MUSIC_TYPE.value}")
print(f"Subjects       : {SUBJECT_INDICES}")
print(f"Pre-onset pad  : {PRE_PAD_S} s")
print(f"Z-score vs rec : {ZSCORE_VS_RECORDING}")
print(f"Plots -> {PLOTS_DIR}")

## Load Concatenated Onsets, Metadata & Channel Names

In [ ]:
safe_label = f"{CONDITION.value}_{MUSIC_TYPE.value}"

concat_dir = (
    ProjectPaths.PROCESSED_DATA_DIR
    / EXPERIMENT.value
    / PreprocessedDataVariants.CONCATENATED.value
)
onsets_path = concat_dir / f"{safe_label}{ProjectPaths.STIMULUS_ONSETS_SUFFIX}"
meta_path = concat_dir / f"{safe_label}.metadata.csv"

wavelet_path = (
    ProjectPaths.PROCESSED_DATA_DIR
    / EXPERIMENT.value
    / "wavelets"
    / "broadband"
    / f"{safe_label}__wavelet_power__{WAVELET_FREQ_SIG}__freqdim1.npz"
)
for p in (onsets_path, meta_path, wavelet_path):
    assert p.exists(), f"Missing expected file: {p}"

onsets = np.load(onsets_path)                        # (n_onsets,) shared sample idx
meta = pd.read_csv(meta_path, index_col=0)
gaps = np.diff(onsets)
print(f"Stimulus onsets : {onsets.shape}  range [{onsets.min()}, {onsets.max()}]")
print(f"Inter-onset gap : min {int(gaps.min())}, median {int(np.median(gaps))}, "
      f"max {int(gaps.max())} samples")

# Channel names are stored in the wavelet feature_names (layout: channel*N_FREQS).
with zipfile.ZipFile(wavelet_path) as _z:
    with _z.open("feature_names.npy") as _f:
        feature_names = npformat.read_array(_f, allow_pickle=True)
channel_names = [str(feature_names[i * N_WAVELET_FREQS]).split("@")[0]
                 for i in range(len(feature_names) // N_WAVELET_FREQS)]
n_channels = len(channel_names)
print(f"Channels parsed : {n_channels} (first={channel_names[0]}, "
      f"last={channel_names[-1]})")
meta[["SingleDataMetadata.PARTICIPANT_ID", "SingleDataMetadata.CONDITION",
      "SingleDataMetadata.CONCATENATED_PERSON_INDEX"]].head()

## Subset Selection & Epoch Window

The post-onset window length is the **shortest inter-onset interval** (`gaps.min()`),
so every epoch fits between its onset and the next one — longer intervals are
trimmed to this shortest gap and no epoch overlaps a neighbouring stimulus. A fixed
`PRE_PAD_S` pad is kept before onset so the map shows the negative-time baseline.

In [ ]:
# Epoch window in samples: fixed pre-onset pad, post length = shortest gap.
PRE = int(round(PRE_PAD_S * SFREQ))
POST = int(gaps.min())                              # trim to shortest inter-onset gap
epoch_times = np.arange(-PRE, POST) / SFREQ         # (win,) seconds, t=0 at onset

# Map subject indices -> participant labels via the concatenated metadata.
pidx_col = "SingleDataMetadata.CONCATENATED_PERSON_INDEX"
pid_col = "SingleDataMetadata.PARTICIPANT_ID"
idx_to_pid = dict(zip(meta[pidx_col], meta[pid_col].astype(str).str.zfill(3)))
# Order subjects by participant ID so every per-participant plot (and the loaded
# trial_avg stack, built from this order) is sorted by PID rather than by
# concatenated index.
SUBJECT_INDICES = sorted(
    SUBJECT_INDICES, key=lambda si: int(idx_to_pid.get(si, "9999"))
)
subject_labels = [f"PSI{idx_to_pid.get(si, '???')}" for si in SUBJECT_INDICES]

print(f"Selected subjects: {dict(zip(SUBJECT_INDICES, subject_labels))}")
print(f"Epoch window     : {PRE + POST} samples ({PRE} pre, {POST} post) "
      f"= [{epoch_times[0]:.3f}, {epoch_times[-1]:.3f}] s")

## Helpers — epoching & lazy trial-averaging reader

In [ ]:
def epoch_average(arr, onsets, pre, post):
    """Average fixed windows around each onset along the LAST axis.

    Args:
        arr: array whose last axis is time, e.g. ``(n_freqs, n_times)``.
        onsets: stimulus onset sample indices.
        pre, post: samples kept before / after each onset (window = pre + post).

    Returns:
        ``(averaged, n_used)`` where the time axis is replaced by the
        ``pre + post`` window, averaged over every onset whose window fits inside
        the recording (edge windows are skipped).
    """
    n_time = arr.shape[-1]
    acc = None
    n_used = 0
    for o in onsets:
        s, e = o - pre, o + post
        if s < 0 or e > n_time:
            continue
        seg = arr[..., s:e]
        acc = seg.astype(np.float64) if acc is None else acc + seg
        n_used += 1
    if n_used == 0:
        raise ValueError("No onset window fits inside the recording.")
    return acc / n_used, n_used


def load_wavelet_trial_averaged(npz_path, subject_indices, n_channels, n_freqs,
                                onsets, pre, post, *, zscore_time=True):
    """Lazily read a huge wavelet npz and trial-average EVERY channel per subject.

    ``savez_compressed`` stores ``data.npy`` as a single deflate stream, so it
    must be decompressed sequentially. This reader streams one
    ``(n_freqs, n_times)`` channel block at a time and immediately reduces it to
    its small ``(n_freqs, win)`` stimulus-locked trial average, so the 49 GB file
    never lands in RAM (peak = one block, ~19 MB). All channels are kept because
    the channel-wise PCA needs the full electrode set.

    Args:
        zscore_time: z-score each channel's per-frequency power against the whole
            recording before epoching (removes the 1/f tilt).

    Returns:
        ``(data, n_times, n_used)`` where ``data`` has shape
        ``(len(subject_indices), n_channels, n_freqs, pre + post)`` and ``n_used``
        maps subject idx -> number of stimuli averaged.
    """
    subj_set = set(subject_indices)
    max_subj = max(subject_indices)
    win = pre + post
    collected = {si: np.empty((n_channels, n_freqs, win)) for si in subject_indices}
    n_used = {}
    with zipfile.ZipFile(npz_path) as z:
        with z.open("data.npy") as f:
            ver = npformat.read_magic(f)
            if ver == (1, 0):
                shape, _, dtype = npformat.read_array_header_1_0(f)
            else:
                shape, _, dtype = npformat.read_array_header_2_0(f)
            n_subj_total, n_feat_flat, n_times = shape
            assert n_feat_flat == n_channels * n_freqs, (
                f"feature axis {n_feat_flat} != n_channels*n_freqs "
                f"{n_channels * n_freqs}"
            )
            block_bytes = n_freqs * n_times * dtype.itemsize  # one channel block
            for s in range(max_subj + 1):
                for c in range(n_channels):
                    buf = f.read(block_bytes)
                    if s not in subj_set:
                        continue
                    block = np.frombuffer(
                        buf, dtype=dtype, count=n_freqs * n_times
                    ).reshape(n_freqs, n_times)
                    if zscore_time:
                        block = zscore(block, axis=1)
                    ev, used = epoch_average(block, onsets, pre, post)  # (f, win)
                    collected[s][c] = ev
                    n_used[s] = used
                if s in subj_set:
                    print(f"  scanned subject block {s} ...")
    data = np.stack([collected[si] for si in subject_indices])
    return data, n_times, n_used


print("Helpers defined.")

## Load & Trial-Average the Wavelet Power

Stream the selected subjects out of the 49 GB cache, reducing every channel to its
stimulus-locked trial average on the fly. This decompresses sequentially from the
start of the file (progress printed per subject block); the result is a small
`(n_subj, n_channels, n_freqs, win)` array.

In [ ]:
wavelet_freqs = np.arange(1.0, N_WAVELET_FREQS + 1.0)   # 1..50 Hz (as cached)

trial_avg, n_times_wav, n_used_w = load_wavelet_trial_averaged(
    wavelet_path,
    subject_indices=SUBJECT_INDICES,
    n_channels=n_channels,
    n_freqs=N_WAVELET_FREQS,
    onsets=onsets,
    pre=PRE,
    post=POST,
    zscore_time=ZSCORE_VS_RECORDING,
)
print(f"Trial-averaged wavelet: {trial_avg.shape}  (n_times_full={n_times_wav})")
for si, label in zip(SUBJECT_INDICES, subject_labels):
    print(f"  {label}: averaged {n_used_w[si]} stimuli")

## Channel PCA — first-component time-frequency map (per subject)

For each subject the trial-averaged data is `(n_channels, n_freqs, win)`. We reshape
it so each `(freq, time)` bin is an **observation** and each channel is a
**variable** — matrix `X` of shape `(n_freqs*win, n_channels)` — and fit PCA. The
**first component's score** for every bin, reshaped to `(n_freqs, win)`, is the TF
map of the dominant spatial mode; this reduces the channel dimension to one map.

PCA component signs are arbitrary, so each map's sign is fixed to make the **ASSR
band (`ASSR_FREQ ± 2` Hz) during the post-onset stimulus interval (`t ≥ 0`)
positive** — i.e. the 40 Hz stimulus response reads **red** on the diverging
`RdBu_r` scale, and the convention is comparable across subjects. The
explained-variance ratio of the first component is reported.

In [ ]:
n_freqs = N_WAVELET_FREQS
win = PRE + POST

# Sign-reference region: the ASSR band (ASSR_FREQ ± 2 Hz) during the post-onset
# stimulus interval (t >= 0). Each map's arbitrary PCA sign is flipped so the mean
# there is POSITIVE, i.e. the 40 Hz stimulus response reads red on RdBu_r.
assr_band = (wavelet_freqs >= ASSR_FREQ - 2) & (wavelet_freqs <= ASSR_FREQ + 2)
post_mask = epoch_times >= 0.0

tf_maps = {}          # subject idx -> (n_freqs, win) first-component TF map
explained = {}        # subject idx -> explained variance ratio of PC1
for k, si in enumerate(SUBJECT_INDICES):
    ep = trial_avg[k]                              # (n_channels, n_freqs, win)
    # Observations = (freq, time) bins, variables = channels.
    X = ep.reshape(n_channels, n_freqs * win).T    # (n_freqs*win, n_channels)
    pca = PCA(n_components=1)
    scores = pca.fit_transform(X)[:, 0]            # (n_freqs*win,)
    tf_map = scores.reshape(n_freqs, win)
    # Fix arbitrary sign: make the ASSR-band post-onset response positive (red).
    ref = tf_map[assr_band][:, post_mask].mean()
    tf_maps[si] = tf_map * np.sign(ref)
    explained[si] = float(pca.explained_variance_ratio_[0])

for si, label in zip(SUBJECT_INDICES, subject_labels):
    print(f"  {label}: PC1 explains {explained[si] * 100:.1f}% of channel variance")

### Per-Participant First-Component TF Map

In [ ]:
# Shared symmetric colour scale across panels for comparability.
vmax = max(np.abs(tf_maps[si]).max() for si in SUBJECT_INDICES)
extent = [epoch_times[0], epoch_times[-1], wavelet_freqs[0], wavelet_freqs[-1]]

fig, axes = plt.subplots(
    1, len(SUBJECT_INDICES), figsize=(5 * len(SUBJECT_INDICES), 4.2), squeeze=False
)
for ax, si, label in zip(axes[0], SUBJECT_INDICES, subject_labels):
    im = ax.imshow(tf_maps[si], aspect="auto", origin="lower", extent=extent,
                   cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.axvline(0.0, color="k", ls="--", lw=0.8)
    ax.axhline(ASSR_FREQ, color="green", ls=":", lw=1.0)
    ax.set_title(f"{label}  (PC1 {explained[si] * 100:.0f}%)")
    ax.set_xlabel("Time rel. onset (s)")
    ax.set_ylabel("Frequency (Hz)")
fig.colorbar(im, ax=axes[0].tolist(), shrink=0.85, label="PC1 score (a.u.)")
fig.suptitle(
    f"Per-participant channel-PCA first-component TF map — "
    f"{CONDITION.value}/{MUSIC_TYPE.value}", y=1.02
)
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "wavelet_pca_tf_per_participant.png", dpi=150,
                bbox_inches="tight")
plt.show()

### First-Component TF Map — averaged over participants

Mean of the per-subject first-component maps. Because each subject's PCA sign is
fixed independently (ASSR-band post-onset response positive), the average is only
meaningful where the dominant spatial mode is consistent across participants.

In [ ]:
group_map = np.mean([tf_maps[si] for si in SUBJECT_INDICES], axis=0)
gvmax = float(np.abs(group_map).max())

fig, ax = plt.subplots(figsize=(7, 4.6))
im = ax.imshow(group_map, aspect="auto", origin="lower", extent=extent,
               cmap="RdBu_r", vmin=-gvmax, vmax=gvmax)
ax.axvline(0.0, color="k", ls="--", lw=0.8, label="onset")
ax.axhline(ASSR_FREQ, color="green", ls=":", lw=1.2, label=f"{ASSR_FREQ:.0f} Hz")
ax.set_title(
    f"Channel-PCA first-component TF map, averaged over participants\n"
    f"{CONDITION.value}/{MUSIC_TYPE.value} (n={len(SUBJECT_INDICES)})"
)
ax.set_xlabel("Time relative to onset (s)")
ax.set_ylabel("Frequency (Hz)")
ax.legend(loc="upper right", fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.9, label="PC1 score (a.u.)")
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "wavelet_pca_tf_group_average.png", dpi=150)
plt.show()